<a href="https://colab.research.google.com/github/Fixer1313/DA_case5/blob/main/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Исследование факторов, влияющих на длительность госпитализации, диагнозы и затраты в медицинских учреждениях. Предварительная обработка

_Как закрепить намертво .csv для привязки к ноутбку в Colab?_

Загрузила файл на гитхаб, дала на него ссылку. Надо проверить, будет ли исчезать

##Импорт библиотек и загрузка датасета

In [1]:
import pandas as pd

In [2]:
url = "https://raw.githubusercontent.com/Fixer1313/DA_case5/refs/heads/main/hospital%20data%20analysis.csv"

In [3]:
df = pd.read_csv(url)
df.head()

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


##Предварительный обзор

###Первичный осмотр данных

In [4]:
print("Форма датасета:", df.shape)
print("\nПервые строки датасета:")
print(df.head())

print("\nИнформация о типах данных:")
print(df.info())

print("\nОписательная статистика:")
print(df[['Age', 'Cost', 'Length_of_Stay', 'Satisfaction']].describe())

Форма датасета: (984, 10)

Первые строки датасета:
   Patient_ID  Age  Gender      Condition                 Procedure   Cost  \
0           1   45  Female  Heart Disease               Angioplasty  15000   
1           2   60    Male       Diabetes           Insulin Therapy   2000   
2           3   32  Female  Fractured Arm          X-Ray and Splint    500   
3           4   75    Male         Stroke    CT Scan and Medication  10000   
4           5   50  Female         Cancer  Surgery and Chemotherapy  25000   

   Length_of_Stay Readmission    Outcome  Satisfaction  
0               5          No  Recovered             4  
1               3         Yes     Stable             3  
2               1          No  Recovered             5  
3               7         Yes     Stable             2  
4              10          No  Recovered             4  

Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 10 columns):
 #   

Датасет содержит информацию о 984 наблюдениях и 10 признаках (5 - числовых, 5 - категориальных). Одна строка соответствует одному случаю госпитализации.
Пропущенные значения на данном этапе не обнаружены. Типы данных соответствуют смыслу переменных. Из описательной статистики исключен признак Patient_ID, как не представляющий интерес для статистической интерпретации.

Предварительно можно сделать вывод, что датасет структурно подготовлен для дальнейшего анализа.

###Просмотр уникальных значений по столбцам

In [5]:
for col in df.columns:
  print(f'Столбец: {col}:')
  print(df[col].value_counts())
  print('-' * 40)

Столбец: Patient_ID:
Patient_ID
1000    1
1       1
2       1
3       1
4       1
       ..
13      1
12      1
11      1
10      1
9       1
Name: count, Length: 984, dtype: int64
----------------------------------------
Столбец: Age:
Age
55    98
45    66
78    66
35    65
52    65
60    65
65    64
40    34
25    34
75    34
68    33
50    33
32    33
70    33
53    33
30    33
62    33
58    33
72    33
48    32
28    32
67    32
Name: count, dtype: int64
----------------------------------------
Столбец: Gender:
Gender
Female    524
Male      460
Name: count, dtype: int64
----------------------------------------
Столбец: Condition:
Condition
Fractured Leg            67
Heart Attack             67
Fractured Arm            66
Hypertension             66
Appendicitis             66
Cancer                   66
Stroke                   66
Allergic Reaction        66
Diabetes                 65
Heart Disease            65
Respiratory Infection    65
Prostate Cancer          65
Childbirth

###Описание переменных

| Переменная | Тип | Описание | Диапазон и количество значений |
|:---|:---|:---|:---|
| **Patient_ID** |  | Идентификатор пациента | 984 уникальных записи |
| **Age** | числовая | Возраст пациента | частичный диапазон от 25 до 78 |
| **Sex** | категориальная | Пол пациента |  2 значения: Female - 524 записи, Male - 460 записей |
| **Condition** | категориальная | Диагноз | 15 значений |
| **Procedure** | категориальная | Проведённое лечение | 15 значений |
| **Cost** | числовая | Стоимость лечения | 15 значений |
| **Length_of_Stay** | числовая | Длительность пребывания в стационаре | ?частичный? диапазон от 1 до 76 |
| **Readmission** | категориальная | Повторная госпитализация | 2 значения: No - 720 записей, Yes - 264 записи |
| **Outcome** | категориальная | Состояние при выписке | 2 значения: Recovered - 591 запись, Stable - 393 записи | | **Satisfaction** | числовая | Оценка пациентом проведённого лечения | диапазон от 2 до 5 |

###Дополнительные изыскания

In [ ]:
print(df['Age'].min(),df['Age'].max())

25 78


##Поиск проблем качества данных

In [6]:
# 1. Дубликаты по всем столбцам и по Patient_ID
print("Полные дубликаты строк:", df.duplicated().sum())
print("Дубликаты Patient_ID:", df['Patient_ID'].duplicated().sum())

Полные дубликаты строк: 0
Дубликаты Patient_ID: 0


In [ ]:
# 2. Проверка пропущенных значений идентификаторов пациентов
expected_ids = set(range(df['Patient_ID'].min(), df['Patient_ID'].max() + 1))
missing_ids = sorted(expected_ids - set(df['Patient_ID']))
print("Пропущенные Patient_ID:", missing_ids)

Пропущенные Patient_ID: [60, 121, 182, 243, 295, 356, 417, 478, 539, 600, 661, 722, 784, 845, 906, 967]


In [ ]:
# 3. Проверка диапазонов
print(df.describe(include='all'))

         Patient_ID         Age  Gender      Condition  \
count    984.000000  984.000000     984            984   
unique          NaN         NaN       2             15   
top             NaN         NaN  Female  Fractured Leg   
freq            NaN         NaN     524             67   
mean     500.329268   53.754065     NaN            NaN   
std      288.979531   14.941135     NaN            NaN   
min        1.000000   25.000000     NaN            NaN   
25%      250.750000   45.000000     NaN            NaN   
50%      500.500000   55.000000     NaN            NaN   
75%      750.250000   65.000000     NaN            NaN   
max     1000.000000   78.000000     NaN            NaN   

                        Procedure          Cost  Length_of_Stay Readmission  \
count                         984    984.000000      984.000000         984   
unique                         15           NaN             NaN           2   
top     Cast and Physical Therapy           NaN             NaN   

In [ ]:
# 4. Связь Condition, Procedure и Cost между собой
# (Уникальных записей в каждом из этих столбцов 15)
unique_per_condition = df.groupby('Condition').agg(
    unique_procedures=('Procedure', 'nunique'),
    unique_costs=('Cost', 'nunique')
)
print(unique_per_condition)

                       unique_procedures  unique_costs
Condition                                             
Allergic Reaction                      1             1
Appendicitis                           1             1
Cancer                                 1             1
Childbirth                             1             1
Diabetes                               1             1
Fractured Arm                          1             1
Fractured Leg                          1             1
Heart Attack                           1             1
Heart Disease                          1             1
Hypertension                           1             1
Kidney Stones                          1             1
Osteoarthritis                         1             1
Prostate Cancer                        1             1
Respiratory Infection                  1             1
Stroke                                 1             1


In [ ]:
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[24, 34, 44, 54, 64, 100],
    labels=['25–34', '35–44', '45–54', '55–64', '65+']
)

Создан дополнительный категориальный признак Age_Group, разделяющий пациентов на возрастные группы для проверки гипотезы о наличии различий в стоимости лечения между пациентами разных возрастных диапазонов.